# Capstone 2: House Price Prediction (Regression)

**Business Problem:** A Real Estate investment firm wants you to build an AI that can predict the exact selling price of a house.

**The Rules:** The training wheels are off. You must write the code from memory or by using your `ML_Cheat_Sheet.md`. I will act as your Senior Developer and review your code.


## Step 1: SQL Extraction & EDA
1. Connect to `real_estate.db`.
2. Write a SQL query that `JOIN`s the `houses` table and the `neighborhoods` table.
3. Load it into a Pandas DataFrame.
4. Do some basic Exploratory Data Analysis (EDA) to understand what columns you have.


In [6]:
# Step 1 Code Here
import sqlite3
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

conn = sqlite3.connect('real_estate.db')

query = """SELECT * FROM houses
JOIN neighborhoods On neighborhoods.neighborhood_id = houses.neighborhood_id
"""
df = pd.read_sql_query(query, conn)
df.head(8)

,house_id,neighborhood_id,sqft,bedrooms,bathrooms,year_built,has_garage,price,neighborhood_id,neighborhood_name,school_rating,crime_rate_index
0,1,1,2396.1,3.0,1.5,1963,Yes,587552.96,1,Downtown,5,8.2
1,2,4,2217.4,3.0,1.5,2021,Yes,539809.82,4,Riverside,6,4.0
2,3,4,2123.2,2.0,1.0,2004,No,491257.02,4,Riverside,6,4.0
3,4,3,NaN,4.0,2.0,1961,No,637377.49,3,University District,7,5.5
4,5,1,1709.6,4.0,2.0,2008,None,519220.21,1,Downtown,5,8.2
5,6,4,2366.5,4.0,2.0,1958,Yes,559870.13,4,Riverside,6,4.0
6,7,2,2209.7,3.0,1.5,1998,None,519633.52,2,Suburbs,9,2.1
7,8,4,1686.5,4.0,2.5,1976,None,415040.72,4,Riverside,6,4.0


## Step 2: The Machine Learning Pipeline
1. Separate your features (`X`) and your target (`y`). **Your target is `price`.**
2. Do a `train_test_split`.
3. Build a `ColumnTransformer` (StandardScaler for your numbers, OneHotEncoder for your text).
4. `fit_transform` your training data.


In [10]:
# Step 2 Code Here
X = df.drop(columns=['price', 'house_id', 'neighborhood_id'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = ['sqft', 'year_built', 'crime_rate_index', 'bedrooms', 'bathrooms', 'school_rating']
cat_cols = ['has_garage', 'neighborhood_name']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## Step 3: Model Training (Regression)
Because we are predicting an exact dollar amount instead of a category, we cannot use a Classifier.

1. Import  from  (instead of ).
2. Import  from .
3. Initialize the  and  it to your processed training data.
4.  the test data and calculate the  between  and .
5. Print the error so we can see how far off our AI is on average!

In [11]:
# Step 3 Code Here
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
model = XGBRegressor(random_state=42)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
error = mean_absolute_error(y_test, y_pred)
print(error)

36794.01899375


## Step 4: Hyperparameter Tuning (GridSearch & CV)
Right now, your model is using the default factory settings. We are going to test multiple settings to see if we can get the error even lower than ,092.

**The Dials we are turning:**
- : How many decision trees to build.
- : How aggressively the AI corrects its mistakes.
- : How deep the trees are allowed to grow (too deep = overfitting).

 will test every combination of these dials 5 times (Cross-Validation) and give you the best one!

In [12]:
# Step 4 Code Here
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'max_depth': [3, 4, 5],
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2]
}

random = RandomizedSearchCV(XGBRegressor(), param_grid, cv=5)

random.fit(X_train_processed, y_train)

print("Best Settings:", random.best_params_)
best_model = random.best_estimator_

Best Settings: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1}


## Step 5: Final Predictions
Now that we have our `best_model` from the Random Search, it`s time to see if we beat our baseline error!

1. Use `best_model.predict()` on your processed testing data.
2. Calculate the `mean_absolute_error` between `y_test` and your new predictions.
3. Print the final error score. Did it beat $26,092?

In [13]:
# Step 5 Code Here
y_pred = best_model.predict(X_test_processed)

error = mean_absolute_error(y_test, y_pred)

print(error)

33766.57965
